# Anomaly Detection in Congressional Stock Trading

Código completo con correcciones para visualización de LOF

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import ttest_ind, pearsonr, gaussian_kde
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import os

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

COLORS = {
    'normal': '#94A3B8',
    'anomaly': '#EF4444',
    'dem': '#3B82F6',
    'rep': '#EF4444',
    'primary': '#475569',
    'secondary': '#94A3B8'
}

FEATURE_LABELS = {
    'disclosure_delay': 'Disclosure Delay',
    'trade_size_mid': 'Trade Size',
    'is_contrarian': 'Contrarian Trade',
    'is_large_trade': 'Large Trade',
    'is_purchase': 'Purchase (vs Sale)',
    'momentum_5d': 'Momentum (5d)',
    'momentum_60d': 'Momentum (60d)',
    'realized_vol_30d': 'Volatility (30d)',
    'volume_ratio_30d': 'Volume Ratio (30d)',
    'abnormal_volume_30d': 'Abnormal Volume (30d)',
    'amihud_illiq_20d': 'Illiquidity (Amihud)',
    'beta_252d': 'Beta (252d)',
    'is_regulated_sector': 'Regulated Sector',
    'is_defense_sector': 'Defense Sector',
    'committee_sector_match': 'Committee-Sector Match',
    'days_since_last_trade': 'Days Since Last Trade',
    'trade_size_vs_history': 'Trade Size vs History',
    'is_new_ticker': 'New Ticker',
    'politician_rolling_win_rate': 'Historical Win Rate',
    'politician_6m_trade_count': 'Trade Frequency (6m)',
    'is_new_sector': 'New Sector',
    'politician_sector_hhi': 'Sector Concentration (HHI)',
    'n_politicians_same_ticker_window': 'Politicians Same Ticker',
    'n_politicians_same_direction': 'Politicians Same Direction',
    'n_same_committee_same_ticker': 'Same Committee Same Ticker',
    'n_same_party_same_ticker': 'Same Party Same Ticker',
    'is_bipartisan_coordination': 'Bipartisan Coordination',
    'coordination_ratio': 'Coordination Ratio',
    'n_politicians_same_sector_week': 'Politicians Same Sector (Week)'
}

def get_label(feature):
    return FEATURE_LABELS.get(feature, feature)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============ CAMBIAR ESTAS RUTAS ============
INPUT_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress\data\congress_trades\congress_trades_anomaly_features.parquet'
OUTPUT_DIR = r'C:\Users\sebib\Documents\GitHub\US_Congress\Parte_2\Anomaly_Detection'
# =============================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('='*70)
print('ANOMALY DETECTION IN CONGRESSIONAL STOCK TRADING')
print('='*70)

## 1. Data Loading

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f'Loaded: {len(df):,} trades')

df = df[df['trade_date'] <= '2025-12-31']
print(f'After filtering: {len(df):,} trades')

FEATURES_POINT = ['disclosure_delay', 'trade_size_mid', 'is_contrarian', 'is_large_trade', 
                  'is_purchase', 'momentum_5d', 'momentum_60d', 'realized_vol_30d', 
                  'volume_ratio_30d', 'abnormal_volume_30d', 'amihud_illiq_20d', 'beta_252d', 
                  'is_regulated_sector', 'is_defense_sector', 'committee_sector_match']
FEATURES_CONTEXTUAL = ['days_since_last_trade', 'trade_size_vs_history', 'is_new_ticker', 
                       'politician_rolling_win_rate', 'politician_6m_trade_count', 
                       'is_new_sector', 'politician_sector_hhi']
FEATURES_COLLECTIVE = ['n_politicians_same_ticker_window', 'n_politicians_same_direction', 
                       'n_same_committee_same_ticker', 'n_same_party_same_ticker', 
                       'is_bipartisan_coordination', 'coordination_ratio', 
                       'n_politicians_same_sector_week']

FEATURES_POINT = [f for f in FEATURES_POINT if f in df.columns]
FEATURES_CONTEXTUAL = [f for f in FEATURES_CONTEXTUAL if f in df.columns]
FEATURES_COLLECTIVE = [f for f in FEATURES_COLLECTIVE if f in df.columns]
FEATURES_ALL = FEATURES_POINT + FEATURES_CONTEXTUAL + FEATURES_COLLECTIVE
print(f'Total features: {len(FEATURES_ALL)}')

df_model = df.dropna(subset=FEATURES_ALL).copy()
for col in FEATURES_ALL:
    df_model[col] = df_model[col].replace([np.inf, -np.inf], np.nan)
    q01, q99 = df_model[col].quantile([0.01, 0.99])
    df_model[col] = df_model[col].clip(q01, q99)
df_model = df_model.dropna(subset=FEATURES_ALL)
print(f'Final sample: {len(df_model):,} trades')

scaler = RobustScaler()
CONTAMINATION = 0.05
MIN_TRADES_LOF = 20

## 2A. Point Anomalies

In [ ]:
print('2A. Point Anomalies...')
X_point = scaler.fit_transform(df_model[FEATURES_POINT])

print('   Training Isolation Forest...')
clf_if = IsolationForest(contamination=CONTAMINATION, n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
df_model['anomaly_if'] = (clf_if.fit_predict(X_point) == -1).astype(int)
df_model['score_if_raw'] = -clf_if.score_samples(X_point)
print(f'   Isolation Forest anomalies: {df_model["anomaly_if"].sum():,}')

print('   Training One-Class SVM...')
idx_sample = np.random.choice(len(X_point), min(20000, len(X_point)), replace=False)
clf_svm = OneClassSVM(nu=CONTAMINATION, kernel='rbf', gamma='auto')
clf_svm.fit(X_point[idx_sample])
df_model['anomaly_svm'] = (clf_svm.predict(X_point) == -1).astype(int)
df_model['score_svm_raw'] = -clf_svm.decision_function(X_point)
print(f'   One-Class SVM anomalies: {df_model["anomaly_svm"].sum():,}')

## 2B. Contextual Anomalies (LOF)

In [ ]:
print('2B. Contextual Anomalies (LOF per politician)...')
FEATURES_FOR_LOF = [f for f in FEATURES_POINT + FEATURES_CONTEXTUAL if f in df_model.columns]

df_model['anomaly_lof_contextual'] = 0
df_model['score_lof_raw'] = np.nan
df_model['lof_computed'] = False

politicians_processed = 0
trades_with_lof = 0

for bioguide_id, group in df_model.groupby('BioGuideID'):
    if len(group) < MIN_TRADES_LOF:
        continue
    
    idx = group.index
    X_pol = RobustScaler().fit_transform(group[FEATURES_FOR_LOF].values)
    n_neighbors = min(20, max(5, len(group) // 2))
    
    if n_neighbors < 5:
        continue
    
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=CONTAMINATION)
    labels = lof.fit_predict(X_pol)
    lof_scores = -lof.negative_outlier_factor_
    
    df_model.loc[idx, 'anomaly_lof_contextual'] = (labels == -1).astype(int)
    df_model.loc[idx, 'score_lof_raw'] = lof_scores
    df_model.loc[idx, 'lof_computed'] = True
    politicians_processed += 1
    trades_with_lof += len(group)

print(f'   Politicians >= {MIN_TRADES_LOF} trades: {politicians_processed}')
print(f'   Trades with LOF: {trades_with_lof:,}')
print(f'   LOF anomalies: {df_model["anomaly_lof_contextual"].sum():,}')

## 2C. Collective Anomalies

In [ ]:
print('2C. Collective Anomalies...')
df_model['anomaly_collective_rules'] = (
    (df_model['n_politicians_same_ticker_window'] >= 3) & 
    (df_model['coordination_ratio'] >= 0.7) & 
    ((df_model['is_bipartisan_coordination'] == 1) | (df_model['n_same_committee_same_ticker'] >= 2))
).astype(int)

X_coll = scaler.fit_transform(df_model[FEATURES_COLLECTIVE])
lof_coll = LocalOutlierFactor(n_neighbors=50, contamination=CONTAMINATION)
df_model['anomaly_lof_collective'] = (lof_coll.fit_predict(X_coll) == -1).astype(int)
df_model['score_lof_collective'] = -lof_coll.negative_outlier_factor_

df_model['anomaly_collective'] = ((df_model['anomaly_collective_rules'] == 1) | 
                                   (df_model['anomaly_lof_collective'] == 1)).astype(int)
print(f'   Collective anomalies: {df_model["anomaly_collective"].sum():,}')

## 3. Ensemble

In [ ]:
print('3. Ensemble...')
anomaly_columns = ['anomaly_if', 'anomaly_svm', 'anomaly_lof_contextual', 'anomaly_collective']
df_model['anomaly_vote_count'] = df_model[anomaly_columns].sum(axis=1)
df_model['anomaly_consensus_2'] = (df_model['anomaly_vote_count'] >= 2).astype(int)

# Normalizar IF, SVM, Collective
for col in ['score_if_raw', 'score_svm_raw', 'score_lof_collective']:
    if col in df_model.columns:
        df_model[f'{col}_norm'] = MinMaxScaler().fit_transform(df_model[[col]])

# Normalizar LOF con P05-P95
lof_mask = df_model['lof_computed'] & df_model['score_lof_raw'].notna()
df_model['score_lof_contextual_norm'] = np.nan

if lof_mask.sum() > 0:
    lof_raw = df_model.loc[lof_mask, 'score_lof_raw'].copy()
    
    # Usar P05 y P95 para definir el rango
    p05 = lof_raw.quantile(0.05)
    p95 = lof_raw.quantile(0.95)
    
    # Normalizar
    normalized = (lof_raw - p05) / (p95 - p05)
    normalized = normalized.clip(0, 1)
    
    df_model.loc[lof_mask, 'score_lof_contextual_norm'] = normalized

print(f'   LOF norm stats: Mean={df_model.loc[lof_mask, "score_lof_contextual_norm"].mean():.3f}')
print(f'   Values at 0: {(df_model.loc[lof_mask, "score_lof_contextual_norm"] == 0).sum()}')
print(f'   Values at 1: {(df_model.loc[lof_mask, "score_lof_contextual_norm"] == 1).sum()}')

# Ensemble score
score_cols = ['score_if_raw_norm', 'score_svm_raw_norm', 'score_lof_contextual_norm', 'score_lof_collective_norm']
score_cols = [c for c in score_cols if c in df_model.columns]
df_model['anomaly_score_ensemble'] = df_model[score_cols].mean(axis=1, skipna=True)

ANOMALY_COL = 'anomaly_consensus_2'
print(f'   Anomalies (>=2 methods): {df_model[ANOMALY_COL].sum():,} ({df_model[ANOMALY_COL].mean():.1%})')

## 4. PCA

In [ ]:
print('4. Computing PCA...')
X_scaled = StandardScaler().fit_transform(df_model[FEATURES_ALL])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_model['PC1'] = np.clip(X_pca[:, 0], -4, 4)
df_model['PC2'] = np.clip(X_pca[:, 1], -4, 4)
print(f'   Variance explained: {pca.explained_variance_ratio_.sum():.1%}')

## 5. Figures

In [ ]:
# FIGURE 1: PCA + Model Agreement
print('Figure 1: PCA + Model Agreement...')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df_sample = df_model.sample(n=min(5000, len(df_model)), random_state=42)
normal_mask = df_sample[ANOMALY_COL] == 0
anomaly_mask = df_sample[ANOMALY_COL] == 1

axes[0].scatter(df_sample.loc[normal_mask, 'PC1'], df_sample.loc[normal_mask, 'PC2'], 
                c=COLORS['normal'], s=5, alpha=0.4, label='Normal')
axes[0].scatter(df_sample.loc[anomaly_mask, 'PC1'], df_sample.loc[anomaly_mask, 'PC2'], 
                c=COLORS['anomaly'], s=10, alpha=0.7, label='Anomaly')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('Anomalies in PCA Space (Consensus >= 2)')
axes[0].legend()

df_lof = df_model[df_model['lof_computed'] & 
                  df_model['score_lof_contextual_norm'].notna() &
                  df_model['score_if_raw_norm'].notna()].copy()

df_lof_sample = df_lof.sample(n=min(3000, len(df_lof)), random_state=42)
normal_lof = df_lof_sample[df_lof_sample[ANOMALY_COL] == 0]
anomaly_lof = df_lof_sample[df_lof_sample[ANOMALY_COL] == 1]

r_corr, _ = pearsonr(df_lof['score_if_raw_norm'], df_lof['score_lof_contextual_norm'])

axes[1].scatter(normal_lof['score_if_raw_norm'], normal_lof['score_lof_contextual_norm'], 
                c=COLORS['normal'], s=5, alpha=0.4, label='Normal')
axes[1].scatter(anomaly_lof['score_if_raw_norm'], anomaly_lof['score_lof_contextual_norm'], 
                c=COLORS['anomaly'], s=10, alpha=0.7, label='Anomaly')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('Isolation Forest Score')
axes[1].set_ylabel('Local Outlier Factor Score')
axes[1].set_title(f'Model Agreement (r = {r_corr:.2f})\n(Politicians with >= {MIN_TRADES_LOF} trades only)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01_pca_model_agreement.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 2: Score Distributions
print('Figure 2: Score Distributions...')
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Distribution of Anomaly Scores', fontsize=14, y=1.02)

# Isolation Forest
axes[0,0].hist(df_model['score_if_raw_norm'], bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
axes[0,0].axvline(df_model['score_if_raw_norm'].quantile(0.95), color=COLORS['anomaly'], linestyle='--', linewidth=2, label='Top 5%')
axes[0,0].set_xlabel('Anomaly Score')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title(f'Isolation Forest\n(n = {len(df_model):,})')
axes[0,0].legend()
axes[0,0].set_xlim(0, 1)

# LOF - excluir valores en 0 y 1 para visualización más limpia
lof_scores = df_model.loc[lof_mask, 'score_lof_contextual_norm']
lof_scores_clean = lof_scores[(lof_scores > 0) & (lof_scores < 1)]
axes[0,1].hist(lof_scores_clean, bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
axes[0,1].axvline(lof_scores_clean.quantile(0.95), color=COLORS['anomaly'], linestyle='--', linewidth=2, label='Top 5%')
axes[0,1].set_xlabel('Anomaly Score')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title(f'Local Outlier Factor\n(n = {len(lof_scores_clean):,} trades)')
axes[0,1].legend()
axes[0,1].set_xlim(0, 1)

# One-Class SVM
axes[1,0].hist(df_model['score_svm_raw_norm'], bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
axes[1,0].axvline(df_model['score_svm_raw_norm'].quantile(0.95), color=COLORS['anomaly'], linestyle='--', linewidth=2, label='Top 5%')
axes[1,0].set_xlabel('Anomaly Score')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title(f'One-Class SVM\n(n = {len(df_model):,})')
axes[1,0].legend()
axes[1,0].set_xlim(0, 1)

# Ensemble
scores_ens = df_model['anomaly_score_ensemble'].dropna()
axes[1,1].hist(scores_ens, bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
axes[1,1].axvline(scores_ens.quantile(0.95), color=COLORS['anomaly'], linestyle='--', linewidth=2, label='Top 5%')
axes[1,1].set_xlabel('Anomaly Score')
axes[1,1].set_ylabel('Frequency')
axes[1,1].set_title(f'Ensemble Score\n(n = {len(scores_ens):,})')
axes[1,1].legend()
axes[1,1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_02_score_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 3: CAR Validation
print('Figure 3: CAR Validation...')
main_car = 'car_ff3_30d' if 'car_ff3_30d' in df_model.columns else 'car_raw_30d'
normal_car = df_model[df_model[ANOMALY_COL]==0][main_car].dropna()
anomaly_car = df_model[df_model[ANOMALY_COL]==1][main_car].dropna()
t_stat, p_val = ttest_ind(anomaly_car, normal_car)
sig = '***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.1 else ''

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Cumulative Abnormal Returns: Normal vs Anomalous Trades', fontsize=14, y=1.02)

box_data = [normal_car, anomaly_car]
bp = axes[0].boxplot(box_data, labels=['Normal', 'Anomaly'], patch_artist=True)
bp['boxes'][0].set_facecolor(COLORS['normal'])
bp['boxes'][1].set_facecolor(COLORS['anomaly'])
axes[0].set_ylabel('CAR (Fama-French 3-Factor, 30d)')
axes[0].set_title(f'CAR Distribution (p = {p_val:.4f} {sig})')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.3)

x_range = np.linspace(min(normal_car.min(), anomaly_car.min()), max(normal_car.max(), anomaly_car.max()), 200)
kde_normal = gaussian_kde(normal_car)
kde_anomaly = gaussian_kde(anomaly_car)
axes[1].fill_between(x_range, kde_normal(x_range), alpha=0.5, color=COLORS['normal'], label=f'Normal (n={len(normal_car):,})')
axes[1].fill_between(x_range, kde_anomaly(x_range), alpha=0.5, color=COLORS['anomaly'], label=f'Anomaly (n={len(anomaly_car):,})')
axes[1].set_xlabel('CAR (Fama-French 3-Factor, 30d)')
axes[1].set_ylabel('Density')
axes[1].set_title('CAR Density Comparison')
axes[1].legend()
axes[1].axvline(0, color='black', linestyle='--', alpha=0.3)

car_cols = [c for c in ['car_ff3_30d', 'car_ff3_60d', 'car_ff3_90d'] if c in df_model.columns]
if len(car_cols) >= 1:
    horizons = ['30d', '60d', '90d'][:len(car_cols)]
    normal_means = [df_model[df_model[ANOMALY_COL]==0][c].mean() for c in car_cols]
    anomaly_means = [df_model[df_model[ANOMALY_COL]==1][c].mean() for c in car_cols]
    x = np.arange(len(horizons))
    width = 0.35
    axes[2].bar(x - width/2, normal_means, width, label='Normal', color=COLORS['normal'])
    axes[2].bar(x + width/2, anomaly_means, width, label='Anomaly', color=COLORS['anomaly'])
    axes[2].set_xlabel('Horizon')
    axes[2].set_ylabel('Mean CAR')
    axes[2].set_title('CAR by Horizon')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(horizons)
    axes[2].legend()
    axes[2].axhline(0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_03_car_validation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nCAR Results:')
print(f'   Normal mean: {normal_car.mean():.4f}')
print(f'   Anomaly mean: {anomaly_car.mean():.4f}')
print(f'   Difference: {anomaly_car.mean() - normal_car.mean():.4f}')
print(f'   t-stat: {t_stat:.2f}, p-value: {p_val:.4f} {sig}')

car_results = []
for horizon, col in zip(['30d', '60d', '90d'], ['car_ff3_30d', 'car_ff3_60d', 'car_ff3_90d']):
    if col in df_model.columns:
        n_car = df_model[df_model[ANOMALY_COL]==0][col].dropna()
        a_car = df_model[df_model[ANOMALY_COL]==1][col].dropna()
        t, p = ttest_ind(a_car, n_car)
        car_results.append({'Horizon': horizon, 'Normal_Mean': n_car.mean(), 
                           'Anomaly_Mean': a_car.mean(), 'Difference': a_car.mean() - n_car.mean(),
                           't_stat': t, 'p_value': p})
car_results_df = pd.DataFrame(car_results)

In [ ]:
# FIGURE 4: Feature Importance
print('Figure 4: Feature Importance...')
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0

normal_df = df_model[df_model[ANOMALY_COL]==0]
anomaly_df = df_model[df_model[ANOMALY_COL]==1]

effect_sizes = {}
for feat in FEATURES_ALL:
    d = cohens_d(anomaly_df[feat], normal_df[feat])
    effect_sizes[feat] = abs(d)

effect_df = pd.DataFrame({'feature': list(effect_sizes.keys()), 'cohens_d': list(effect_sizes.values())})
effect_df = effect_df.sort_values('cohens_d', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(effect_df['feature'].apply(get_label), effect_df['cohens_d'], color=COLORS['anomaly'], edgecolor='white')
ax.set_xlabel("Cohen's d (Effect Size)")
ax.set_title('Feature Importance: Anomalous vs Normal Trades')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_04_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 5: Anomaly by Party/Chamber/Role
print('Figure 5: Anomaly by Institution...')
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

if 'Party' in df_model.columns:
    party_rates = df_model.groupby('Party')[ANOMALY_COL].mean() * 100
    colors_party = [COLORS['dem'] if p=='D' else COLORS['rep'] if p=='R' else COLORS['secondary'] for p in party_rates.index]
    axes[0].bar(party_rates.index, party_rates.values, color=colors_party, edgecolor='white')
    axes[0].set_ylabel('Anomaly Rate (%)')
    axes[0].set_title('By Party')
    for i, v in enumerate(party_rates.values):
        axes[0].text(i, v + 0.1, f'{v:.1f}%', ha='center')

if 'Chamber' in df_model.columns:
    chamber_rates = df_model.groupby('Chamber')[ANOMALY_COL].mean() * 100
    axes[1].bar(chamber_rates.index, chamber_rates.values, color=[COLORS['primary'], COLORS['secondary']], edgecolor='white')
    axes[1].set_ylabel('Anomaly Rate (%)')
    axes[1].set_title('By Chamber')
    for i, v in enumerate(chamber_rates.values):
        axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center')

if 'is_chair_or_ranking' in df_model.columns:
    df_model['role'] = df_model['is_chair_or_ranking'].map({0: 'Regular Member', 1: 'Chair/Ranking'})
    role_rates = df_model.groupby('role')[ANOMALY_COL].mean() * 100
    axes[2].bar(role_rates.index, role_rates.values, color=[COLORS['secondary'], COLORS['primary']], edgecolor='white')
    axes[2].set_ylabel('Anomaly Rate (%)')
    axes[2].set_title('By Committee Role')
    for i, v in enumerate(role_rates.values):
        axes[2].text(i, v + 0.1, f'{v:.1f}%', ha='center')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_05_anomaly_by_institution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 6: Anomaly by Committee
print('Figure 6: Anomaly by Committee...')
if 'committee_topic' in df_model.columns:
    comm_stats = df_model.groupby('committee_topic').agg({ANOMALY_COL: ['sum', 'count', 'mean']}).reset_index()
    comm_stats.columns = ['committee', 'n_anomalies', 'n_trades', 'anomaly_rate']
    comm_stats = comm_stats[comm_stats['n_trades'] >= 100].sort_values('anomaly_rate', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(comm_stats['committee'], comm_stats['anomaly_rate']*100, color=COLORS['anomaly'], edgecolor='white')
    ax.set_xlabel('Anomaly Rate (%)')
    ax.set_title('Anomaly Rate by Committee Topic')
    ax.axvline(df_model[ANOMALY_COL].mean()*100, color='black', linestyle='--', alpha=0.5, label='Overall average')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_06_anomaly_by_committee.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
# FIGURE 7: Anomaly by Sector
print('Figure 7: Anomaly by Sector...')
if 'sector' in df_model.columns:
    sector_stats = df_model.groupby('sector').agg({ANOMALY_COL: ['sum', 'count', 'mean']}).reset_index()
    sector_stats.columns = ['sector', 'n_anomalies', 'n_trades', 'anomaly_rate']
    sector_stats = sector_stats[sector_stats['n_trades'] >= 100].sort_values('anomaly_rate', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(sector_stats['sector'], sector_stats['anomaly_rate']*100, color=COLORS['anomaly'], edgecolor='white')
    ax.set_xlabel('Anomaly Rate (%)')
    ax.set_title('Anomaly Rate by Stock Sector')
    ax.axvline(df_model[ANOMALY_COL].mean()*100, color='black', linestyle='--', alpha=0.5, label='Overall average')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_07_anomaly_by_sector.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
# FIGURE 8: Anomaly Over Time
print('Figure 8: Anomaly Over Time...')
df_model['trade_year'] = pd.to_datetime(df_model['trade_date']).dt.year
yearly_stats = df_model.groupby('trade_year').agg({ANOMALY_COL: ['sum', 'count', 'mean']}).reset_index()
yearly_stats.columns = ['year', 'n_anomalies', 'n_trades', 'anomaly_rate']

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_stats['year'], yearly_stats['anomaly_rate']*100, 'o-', color=COLORS['anomaly'], linewidth=2, markersize=8)
ax.axvline(2020, color='gray', linestyle='--', alpha=0.7, label='COVID Trading Scandal')
ax.set_xlabel('Year')
ax.set_ylabel('Anomaly Rate (%)')
ax.set_title('Anomaly Rate Over Time')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_08_anomaly_over_time.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 9: Top Politicians
print('Figure 9: Top Politicians...')
politician_stats = df_model.groupby(['BioGuideID', 'Name', 'Party']).agg({
    ANOMALY_COL: ['sum', 'count', 'mean'],
    'car_ff3_30d': 'mean' if 'car_ff3_30d' in df_model.columns else 'count'
}).reset_index()
politician_stats.columns = ['BioGuideID', 'Name', 'Party', 'n_anomalies', 'n_trades', 'anomaly_rate', 'mean_car']
politician_stats = politician_stats[politician_stats['n_trades'] >= 50].sort_values('anomaly_rate', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 8))
colors = [COLORS['dem'] if p=='D' else COLORS['rep'] for p in politician_stats['Party']]
ax.barh(politician_stats['Name'], politician_stats['anomaly_rate']*100, color=colors, edgecolor='white')
ax.set_xlabel('Anomaly Rate (%)')
ax.set_title('Top 15 Politicians by Anomaly Rate (min 50 trades)')
ax.axvline(df_model[ANOMALY_COL].mean()*100, color='black', linestyle='--', alpha=0.5, label='Overall average')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_09_top_politicians.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# FIGURE 10: Pre vs Post 2020
print('Figure 10: Pre vs Post 2020...')
pre = df_model[df_model['trade_year']<2020][ANOMALY_COL].mean()*100
post = df_model[df_model['trade_year']>=2020][ANOMALY_COL].mean()*100

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(['Pre-2020', 'Post-2020'], [pre, post], color=[COLORS['normal'], COLORS['anomaly']], edgecolor='white')
ax.set_ylabel('Anomaly Rate (%)')
ax.set_title('Anomaly Rate: Pre vs Post 2020')
for i, v in enumerate([pre, post]):
    ax.text(i, v + 0.1, f'{v:.2f}%', ha='center', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_10_pre_post_2020.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Export

In [ ]:
print('6. Exporting results...')
out_cols = ['trade_id', 'BioGuideID', 'Name', 'Party', 'Chamber', 'Ticker', 'trade_date', 
            'Transaction', 'Trade_Size_USD', 'sector', 'anomaly_if', 'anomaly_svm', 
            'anomaly_lof_contextual', 'anomaly_collective', 'anomaly_vote_count', 
            'anomaly_consensus_2', 'anomaly_score_ensemble', 'car_ff3_30d', 'car_ff3_60d', 'car_ff3_90d']
out_cols = [c for c in out_cols if c in df_model.columns]
df_model[out_cols].to_csv(f'{OUTPUT_DIR}/anomaly_detection_results.csv', index=False)
politician_stats.to_csv(f'{OUTPUT_DIR}/politician_anomaly_summary.csv', index=False)
car_results_df.to_csv(f'{OUTPUT_DIR}/car_validation_results.csv', index=False)
print(f'   Saved to: {OUTPUT_DIR}')

print('\n' + '='*70)
print('SUMMARY')
print('='*70)
print(f'Total trades: {len(df_model):,}')
print(f'Trades with LOF: {df_model["lof_computed"].sum():,} ({df_model["lof_computed"].mean():.1%})')
print(f'Anomalies (>=2 methods): {df_model[ANOMALY_COL].sum():,} ({df_model[ANOMALY_COL].mean():.1%})')
if 'car_ff3_30d' in df_model.columns:
    print(f'CAR Normal: {df_model[df_model[ANOMALY_COL]==0]["car_ff3_30d"].mean():.4f}')
    print(f'CAR Anomaly: {df_model[df_model[ANOMALY_COL]==1]["car_ff3_30d"].mean():.4f}')
print(f'\nFigures saved: fig_01 to fig_10')
print('='*70)